In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# ==========================================
# E-COMMERCE DATA ENGINEERING
# SILVER LAYER - SAFE DATA CLEANING
# ==========================================

SCHEMA = "workspace.ecommerce"

tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

timestamp_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

numeric_columns = {
    "geolocation": {
        "geolocation_lat": "DOUBLE",
        "geolocation_lng": "DOUBLE"
    },
    "order_items": {
        "order_item_id": "INT",
        "price": "DECIMAL(18,2)",
        "freight_value": "DECIMAL(18,2)"
    },
    "order_payments": {
        "payment_sequential": "INT",
        "payment_installments": "INT",
        "payment_value": "DECIMAL(18,2)"
    },
    "order_reviews": {
        "review_score": "INT"
    },
    "products": {
        "product_name_lenght": "INT",
        "product_description_lenght": "INT",
        "product_photos_qty": "INT",
        "product_weight_g": "INT",
        "product_length_cm": "INT",
        "product_height_cm": "INT",
        "product_width_cm": "INT"
    }
}

results = []

for table in tables:

    print(f"Processing: {table}")

    source_table = f"{SCHEMA}.bronze_{table}"
    target_table = f"{SCHEMA}.silver_{table}"

    df = spark.table(source_table)

    # 1. Standardise column names
    for old_name in df.columns:
        new_name = old_name.strip().lower().replace(" ", "_")
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    # 2. Trim text columns
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(
                field.name,
                F.trim(F.col(field.name))
            )

    # 3. Remove exact duplicate records
    bronze_count = df.count()
    df = df.dropDuplicates()
    deduplicated_count = df.count()

    # 4. Safely convert timestamps
    for col_name in timestamp_columns.get(table, []):
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                F.expr(
                    f"try_to_timestamp(`{col_name}`)"
                )
            )

    # 5. Safely convert numeric columns
    for col_name, data_type in numeric_columns.get(table, {}).items():
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                F.expr(
                    f"try_cast(`{col_name}` AS {data_type})"
                )
            )

    # 6. Add Silver processing metadata
    df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    # 7. Write cleaned Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    silver_count = spark.table(target_table).count()

    results.append((
        table,
        bronze_count,
        deduplicated_count,
        silver_count,
        bronze_count - deduplicated_count
    ))

    print(
        f"SUCCESS: {target_table} | "
        f"Rows: {silver_count} | "
        f"Duplicates removed: {bronze_count - deduplicated_count}"
    )

# 8. Create processing summary
summary_df = spark.createDataFrame(
    results,
    [
        "dataset",
        "bronze_rows",
        "deduplicated_rows",
        "silver_rows",
        "duplicates_removed"
    ]
)

print("======================================")
print("SILVER TRANSFORMATION COMPLETED")
print("======================================")

display(summary_df)



Processing: customers
SUCCESS: workspace.ecommerce.silver_customers | Rows: 99441 | Duplicates removed: 0
Processing: geolocation
SUCCESS: workspace.ecommerce.silver_geolocation | Rows: 738332 | Duplicates removed: 261831
Processing: order_items
SUCCESS: workspace.ecommerce.silver_order_items | Rows: 112650 | Duplicates removed: 0
Processing: order_payments
SUCCESS: workspace.ecommerce.silver_order_payments | Rows: 103886 | Duplicates removed: 0
Processing: order_reviews
SUCCESS: workspace.ecommerce.silver_order_reviews | Rows: 104067 | Duplicates removed: 95
Processing: orders
SUCCESS: workspace.ecommerce.silver_orders | Rows: 99441 | Duplicates removed: 0
Processing: products
SUCCESS: workspace.ecommerce.silver_products | Rows: 32951 | Duplicates removed: 0
Processing: sellers
SUCCESS: workspace.ecommerce.silver_sellers | Rows: 3095 | Duplicates removed: 0
Processing: category_translation
SUCCESS: workspace.ecommerce.silver_category_translation | Rows: 71 | Duplicates removed: 0
SILV

dataset,bronze_rows,deduplicated_rows,silver_rows,duplicates_removed
customers,99441,99441,99441,0
geolocation,1000163,738332,738332,261831
order_items,112650,112650,112650,0
order_payments,103886,103886,103886,0
order_reviews,104162,104067,104067,95
orders,99441,99441,99441,0
products,32951,32951,32951,0
sellers,3095,3095,3095,0
category_translation,71,71,71,0
